In [1]:
# ============================================================
# AMP Challenge — Multi-Batch Clean Pool Builder v2
# ============================================================
# Purpose:
#   Merge multiple generated FASTA batches (e.g., 5 x 50K),
#   remove invalid / duplicate / disallowed candidates,
#   preserve provenance, and build ONE clean candidate pool.
#
# IMPORTANT:
#   This notebook does NOT select the final 50K and does NOT select Top-100.
#   Final selection will be done later using:
#   activity + safety + physicochemical properties + diversity + novelty.
#
# Screening rules in this notebook:
# 1) Generated candidates
#    - remove empty sequences
#    - remove sequences containing non-standard amino acids
#    - remove sequences outside length 8–50
#    - remove exact duplicates across ALL uploaded batches
#
# 2) Official Challenge reference
#    - filename must contain "antibacterial"
#    - exact match -> REMOVE
#    - RapidFuzz fuzz.ratio > 80% -> REMOVE
#    - exactly 80% -> KEEP at this pre-screening stage
#
# 3) External AMP databases
#    - e.g. DBAASP, dbAMP3, APD/APD6, DRAMP
#    - exact sequence overlap -> REMOVE
#    - near matches are NOT removed in this notebook
#
# NOTE:
#   RapidFuzz fuzz.ratio is a Levenshtein/Indel-style similarity pre-screen.
#   Before final competition submission, the official Challenge validation /
#   seqme implementation should be run for the official identity requirement.
#
# Outputs:
# - FULL_AUDIT.csv
# - REMOVED.csv
# - ALL_CLEAN.csv
# - ALL_CLEAN.fasta
# - BATCH_SUMMARY.csv
# - SCREENING_MANIFEST.txt
# ============================================================

!pip -q install rapidfuzz pandas tqdm

from google.colab import files
from rapidfuzz import process, fuzz
from tqdm.auto import tqdm
from collections import defaultdict
from datetime import datetime
import pandas as pd
import os
import re
import math

# ============================================================
# CONFIGURATION
# ============================================================

OFFICIAL_THRESHOLD = 80.0
MIN_LENGTH = 8
MAX_LENGTH = 50
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# ============================================================
# HELPERS
# ============================================================

def read_fasta(filename):
    """
    Read FASTA while preserving headers and also retaining empty records
    so they can be explicitly audited as EMPTY_SEQUENCE.
    """
    records = []
    current_id = None
    seq_parts = []

    def flush_record():
        nonlocal current_id, seq_parts
        if current_id is not None:
            seq = "".join(seq_parts).upper().strip()
            records.append((current_id, seq))

    with open(filename, "r", encoding="utf-8", errors="ignore") as f:
        for raw_line in f:
            line = raw_line.strip()

            if line.startswith(">"):
                flush_record()
                current_id = line[1:].strip() or f"unnamed_{len(records)+1}"
                seq_parts = []
            elif line:
                seq_parts.append(re.sub(r"\s+", "", line))

        flush_record()

    return records


def clean_name(filename):
    name = os.path.basename(filename)
    for ext in [".fasta", ".fa", ".faa", ".fas", ".txt", ".csv", ".tsv"]:
        if name.lower().endswith(ext):
            name = name[:-len(ext)]
            break
    return re.sub(r"[^A-Za-z0-9_-]+", "_", name).strip("_")


# ============================================================
# 1. UPLOAD GENERATED BATCHES
# ============================================================

print("\n==============================")
print("UPLOAD GENERATED FASTA BATCHES")
print("==============================")
print("Upload ALL generated batches together (for example, 5 HydrAMP batches).\n")

generated_upload = files.upload()
generated_files = list(generated_upload.keys())

if not generated_files:
    raise ValueError("No generated FASTA file was uploaded.")

print("\nGenerated files:")
for f in generated_files:
    print(" -", f)


# ============================================================
# 2. UPLOAD REFERENCE DATABASES
# ============================================================

print("\n==============================")
print("UPLOAD REFERENCE DATABASES")
print("==============================")
print(
    "Recommended:\n"
    "- antibacterial.fasta  (official Challenge reference)\n"
    "- DBAASP.fasta\n"
    "- dbAMP3.fasta\n"
    "- ADP6.fasta / APD6.fasta\n"
    "- DRAMP.fasta (optional)\n"
)

reference_upload = files.upload()
reference_files = list(reference_upload.keys())

if not reference_files:
    raise ValueError("No reference database was uploaded.")


# ============================================================
# 3. IDENTIFY OFFICIAL CHALLENGE REFERENCE
# ============================================================

official_candidates = [
    f for f in reference_files
    if "antibacterial" in os.path.basename(f).lower()
]

if len(official_candidates) != 1:
    raise ValueError(
        "Exactly one uploaded reference filename must contain "
        "'antibacterial' so the official Challenge reference "
        "can be identified unambiguously."
    )

official_file = official_candidates[0]
external_files = [f for f in reference_files if f != official_file]

print("\nOfficial Challenge reference:")
print(" ", official_file)

print("\nExternal novelty databases:")
for f in external_files:
    print(" ", f)


# ============================================================
# 4. LOAD GENERATED BATCHES + PRESERVE PROVENANCE
# ============================================================

generated_records = []

for upload_order, filename in enumerate(generated_files, start=1):
    batch_name = clean_name(filename)
    records = read_fasta(filename)

    for within_batch_index, (record_id, sequence) in enumerate(records, start=1):
        generated_records.append({
            "upload_order": upload_order,
            "batch": batch_name,
            "source_filename": os.path.basename(filename),
            "within_batch_index": within_batch_index,
            "original_id": record_id,
            "sequence": sequence.upper().strip()
        })

generated_df = pd.DataFrame(generated_records)

print("\n==============================")
print("GENERATED DATA SUMMARY")
print("==============================")
print(f"Files uploaded            : {len(generated_files):,}")
print(f"Total sequences loaded    : {len(generated_df):,}")

input_batch_summary = (
    generated_df.groupby(["upload_order", "batch", "source_filename"])
    .size()
    .reset_index(name="input_count")
    .sort_values("upload_order")
)

display(input_batch_summary)


# ============================================================
# 5. LOAD REFERENCES
# ============================================================

reference_data = {}

for filename in reference_files:
    reference_data[filename] = read_fasta(filename)

print("\n==============================")
print("REFERENCE DATABASE SUMMARY")
print("==============================")

for filename, records in reference_data.items():
    seqs = [seq.upper().strip() for _, seq in records if seq.strip()]
    print(
        f"{clean_name(filename):20s} "
        f"| total={len(records):7d} "
        f"| unique={len(set(seqs)):7d}"
    )


# ============================================================
# 6. PREPARE OFFICIAL REFERENCE
# ============================================================

official_seq_to_id = {}

for ref_id, seq in reference_data[official_file]:
    seq = seq.upper().strip()
    if seq and seq not in official_seq_to_id:
        official_seq_to_id[seq] = ref_id

official_sequences = list(official_seq_to_id.keys())
official_exact_set = set(official_sequences)

official_length_buckets = defaultdict(list)
for seq in official_sequences:
    official_length_buckets[len(seq)].append(seq)


# ============================================================
# 7. PREPARE EXTERNAL EXACT-MATCH SETS
# ============================================================

external_sets = {}
external_ids = {}

for filename in external_files:
    seq_set = set()
    seq_to_id = {}

    for ref_id, seq in reference_data[filename]:
        seq = seq.upper().strip()
        if not seq:
            continue

        seq_set.add(seq)
        if seq not in seq_to_id:
            seq_to_id[seq] = ref_id

    external_sets[filename] = seq_set
    external_ids[filename] = seq_to_id


# ============================================================
# 8. SCREEN COMBINED GENERATED POOL
# ============================================================

results = []
seen_sequences = set()

for row in tqdm(
    generated_df.itertuples(index=False),
    total=len(generated_df),
    desc="Screening candidates"
):
    sequence = row.sequence.upper().strip()

    result = {
        "upload_order": row.upload_order,
        "batch": row.batch,
        "source_filename": row.source_filename,
        "within_batch_index": row.within_batch_index,
        "original_id": row.original_id,
        "sequence": sequence,
        "length": len(sequence),

        "valid_alphabet": True,
        "valid_length": True,
        "internal_duplicate": False,

        "official_exact_match": False,
        "official_similarity_pct": None,
        "official_match_id": None,
        "official_match_sequence": None,

        "external_exact_overlap": False,
        "external_database": None,
        "external_match_id": None,

        "keep": True,
        "removal_reason": "PASS"
    }

    # EMPTY
    if not sequence:
        result["keep"] = False
        result["removal_reason"] = "EMPTY_SEQUENCE"
        results.append(result)
        continue

    # STANDARD AMINO-ACID ALPHABET
    invalid = set(sequence) - STANDARD_AA
    if invalid:
        result["valid_alphabet"] = False
        result["keep"] = False
        result["removal_reason"] = "INVALID_AA_" + "".join(sorted(invalid))
        results.append(result)
        continue

    # LENGTH
    if not MIN_LENGTH <= len(sequence) <= MAX_LENGTH:
        result["valid_length"] = False
        result["keep"] = False
        result["removal_reason"] = "INVALID_LENGTH"
        results.append(result)
        continue

    # DUPLICATES ACROSS THE COMPLETE MULTI-BATCH POOL
    if sequence in seen_sequences:
        result["internal_duplicate"] = True
        result["keep"] = False
        result["removal_reason"] = "INTERNAL_DUPLICATE"
        results.append(result)
        continue

    seen_sequences.add(sequence)

    # OFFICIAL EXACT MATCH
    if sequence in official_exact_set:
        result["official_exact_match"] = True
        result["official_similarity_pct"] = 100.0
        result["official_match_id"] = official_seq_to_id[sequence]
        result["official_match_sequence"] = sequence
        result["keep"] = False
        result["removal_reason"] = "OFFICIAL_EXACT_MATCH"
        results.append(result)
        continue

    # OFFICIAL >80% PRE-SCREEN
    L = len(sequence)

    # For RapidFuzz fuzz.ratio, references outside this length interval
    # cannot exceed the 80% cutoff even under a best-case subsequence match.
    min_ref_len = max(1, math.ceil(L * 2 / 3))
    max_ref_len = math.floor(L * 1.5)

    candidate_refs = []
    for ref_len in range(min_ref_len, max_ref_len + 1):
        candidate_refs.extend(official_length_buckets.get(ref_len, []))

    if candidate_refs:
        best_hit = process.extractOne(
            sequence,
            candidate_refs,
            scorer=fuzz.ratio,
            score_cutoff=OFFICIAL_THRESHOLD
        )
    else:
        best_hit = None

    if best_hit:
        matched_seq, similarity, _ = best_hit

        result["official_similarity_pct"] = round(float(similarity), 3)
        result["official_match_sequence"] = matched_seq
        result["official_match_id"] = official_seq_to_id.get(matched_seq)

        # Challenge-oriented conservative pre-screen:
        # >80 is removed, exactly 80 is retained here.
        if similarity > OFFICIAL_THRESHOLD:
            result["keep"] = False
            result["removal_reason"] = "OFFICIAL_ABOVE_80"
            results.append(result)
            continue

    # EXTERNAL EXACT OVERLAP
    for filename in external_files:
        if sequence in external_sets[filename]:
            result["external_exact_overlap"] = True
            result["external_database"] = clean_name(filename)
            result["external_match_id"] = external_ids[filename].get(sequence)
            result["keep"] = False
            result["removal_reason"] = "EXACT_OVERLAP_" + clean_name(filename)
            break

    results.append(result)


# ============================================================
# 9. BUILD AUDIT TABLES
# ============================================================

audit_df = pd.DataFrame(results)
clean_df = audit_df[audit_df["keep"] == True].copy()
removed_df = audit_df[audit_df["keep"] == False].copy()

# Stable clean-pool order:
# keep original upload order and within-batch order.
# We deliberately DO NOT call this a "BEST_50K" ranking.
clean_df = (
    clean_df
    .sort_values(["upload_order", "within_batch_index"])
    .reset_index(drop=True)
)

clean_df["clean_pool_id"] = [
    f"CLEAN_{i:07d}" for i in range(1, len(clean_df) + 1)
]


# ============================================================
# 10. PER-BATCH AUDIT SUMMARY
# ============================================================

batch_rows = []

for upload_order, batch, source_filename in (
    generated_df[["upload_order", "batch", "source_filename"]]
    .drop_duplicates()
    .sort_values("upload_order")
    .itertuples(index=False, name=None)
):
    sub = audit_df[audit_df["batch"] == batch]

    batch_rows.append({
        "upload_order": upload_order,
        "batch": batch,
        "source_filename": source_filename,
        "input_count": len(sub),
        "kept_count": int(sub["keep"].sum()),
        "removed_count": int((~sub["keep"]).sum()),
        "internal_duplicates": int(sub["internal_duplicate"].sum()),
        "official_exact_matches": int(sub["official_exact_match"].sum()),
        "official_above_80": int((sub["removal_reason"] == "OFFICIAL_ABOVE_80").sum()),
        "external_exact_overlaps": int(sub["external_exact_overlap"].sum()),
        "retention_pct": round((sub["keep"].mean() * 100.0), 3) if len(sub) else 0.0
    })

batch_summary_df = pd.DataFrame(batch_rows)


# ============================================================
# 11. OUTPUT PREFIX
# ============================================================

# Keep filenames short even when 5+ batches are uploaded.
reference_tag = "__".join(sorted(clean_name(f) for f in reference_files))
prefix = f"MULTIBATCH_{len(generated_files)}__filtered_against__{reference_tag}"

audit_file = prefix + "__FULL_AUDIT.csv"
removed_file = prefix + "__REMOVED.csv"
clean_file = prefix + "__ALL_CLEAN.csv"
clean_fasta = prefix + "__ALL_CLEAN.fasta"
batch_summary_file = prefix + "__BATCH_SUMMARY.csv"
manifest_file = prefix + "__SCREENING_MANIFEST.txt"


# ============================================================
# 12. SAVE CSV OUTPUTS
# ============================================================

audit_df.to_csv(audit_file, index=False)
removed_df.to_csv(removed_file, index=False)
clean_df.to_csv(clean_file, index=False)
batch_summary_df.to_csv(batch_summary_file, index=False)


# ============================================================
# 13. SAVE CLEAN FASTA
# ============================================================

with open(clean_fasta, "w") as f:
    for row in clean_df.itertuples(index=False):
        f.write(
            f">{row.clean_pool_id}"
            f"|batch={row.batch}"
            f"|original_id={row.original_id}\n"
        )
        f.write(row.sequence + "\n")


# ============================================================
# 14. SAVE SCREENING MANIFEST
# ============================================================

with open(manifest_file, "w") as f:
    f.write("AMP Challenge — Multi-Batch Clean Pool Builder v2\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Run timestamp: {datetime.now().isoformat(timespec='seconds')}\n")
    f.write(f"Generated files: {len(generated_files)}\n")
    for x in generated_files:
        f.write(f"  - {x}\n")

    f.write("\nOfficial Challenge reference:\n")
    f.write(f"  - {official_file}\n")

    f.write("\nExternal exact-overlap references:\n")
    for x in external_files:
        f.write(f"  - {x}\n")

    f.write("\nConfiguration:\n")
    f.write(f"  MIN_LENGTH={MIN_LENGTH}\n")
    f.write(f"  MAX_LENGTH={MAX_LENGTH}\n")
    f.write(f"  OFFICIAL_THRESHOLD={OFFICIAL_THRESHOLD}\n")
    f.write("  STANDARD_AA=ACDEFGHIKLMNPQRSTVWY\n")

    f.write("\nImportant interpretation:\n")
    f.write(
        "RapidFuzz fuzz.ratio is used only as a similarity pre-screen. "
        "Final official identity validation should be repeated using the "
        "competition's official validation/seqme implementation.\n"
    )
    f.write(
        "This notebook creates a clean candidate pool only. "
        "It does not select the final 50,000 library or Top-100.\n"
    )


# ============================================================
# 15. SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("AMP CHALLENGE MULTI-BATCH CLEANING SUMMARY")
print("=" * 70)

print(f"Generated batches uploaded : {len(generated_files):,}")
print(f"Input sequences            : {len(audit_df):,}")
print(f"Unique clean sequences     : {len(clean_df):,}")
print(f"Removed sequences          : {len(removed_df):,}")

if len(audit_df):
    print(f"Overall retention rate     : {len(clean_df)/len(audit_df)*100:.2f}%")

print("\nPER-BATCH SUMMARY")
display(batch_summary_df)

print("\nREMOVAL REASONS")
removal_summary = (
    audit_df["removal_reason"]
    .value_counts()
    .rename_axis("reason")
    .reset_index(name="count")
)
display(removal_summary)

print("\nEXTERNAL EXACT-OVERLAP BREAKDOWN")
external_breakdown = (
    audit_df[audit_df["external_exact_overlap"] == True]["external_database"]
    .value_counts()
    .rename_axis("database")
    .reset_index(name="exact_overlaps")
)
display(external_breakdown)

print("\n")
print("=" * 70)
print("IMPORTANT")
print("=" * 70)
print("This run DOES NOT create BEST_50K or final Top-100.")
print("ALL_CLEAN is the candidate pool for the next scoring/selection stages.")
print("Final selection will use activity, safety, physicochemical,")
print("diversity and novelty information together.")


# ============================================================
# 16. OUTPUT FILES
# ============================================================

print("\nOUTPUT FILES")
print(" -", audit_file)
print(" -", removed_file)
print(" -", clean_file)
print(" -", clean_fasta)
print(" -", batch_summary_file)
print(" -", manifest_file)


# ============================================================
# 17. DOWNLOAD OUTPUTS
# ============================================================

print("\nStarting downloads...")

for output_file in [
    audit_file,
    removed_file,
    clean_file,
    clean_fasta,
    batch_summary_file,
    manifest_file
]:
    files.download(output_file)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 20.2 MB/s eta 0:00:00

UPLOAD GENERATED FASTA BATCHES
Upload ALL generated batches together (for example, 5 HydrAMP batches).



Saving hydramp_batch1_seed42.fasta to hydramp_batch1_seed42.fasta
Saving hydramp_batch2_seed44.fasta to hydramp_batch2_seed44.fasta
Saving hydramp_batch3_seed46.fasta to hydramp_batch3_seed46.fasta
Saving hydramp_batch4_seed48.fasta to hydramp_batch4_seed48.fasta
Saving hydramp_batch5_seed50.fasta to hydramp_batch5_seed50.fasta

Generated files:
 - hydramp_batch1_seed42.fasta
 - hydramp_batch2_seed44.fasta
 - hydramp_batch3_seed46.fasta
 - hydramp_batch4_seed48.fasta
 - hydramp_batch5_seed50.fasta

UPLOAD REFERENCE DATABASES
Recommended:
- antibacterial.fasta  (official Challenge reference)
- DBAASP.fasta
- dbAMP3.fasta
- ADP6.fasta / APD6.fasta
- DRAMP.fasta (optional)



Saving ADP6.fasta to ADP6.fasta
Saving antibacterial.fasta to antibacterial.fasta
Saving DBAASP.fasta to DBAASP.fasta
Saving dbAMP3.fasta to dbAMP3.fasta

Official Challenge reference:
  antibacterial.fasta

External novelty databases:
  ADP6.fasta
  DBAASP.fasta
  dbAMP3.fasta

GENERATED DATA SUMMARY
Files uploaded            : 5
Total sequences loaded    : 250,000


,upload_order,batch,source_filename,input_count
0,1,hydramp_batch1_seed42,hydramp_batch1_seed42.fasta,50000
1,2,hydramp_batch2_seed44,hydramp_batch2_seed44.fasta,50000
2,3,hydramp_batch3_seed46,hydramp_batch3_seed46.fasta,50000
3,4,hydramp_batch4_seed48,hydramp_batch4_seed48.fasta,50000
4,5,hydramp_batch5_seed50,hydramp_batch5_seed50.fasta,50000



REFERENCE DATABASE SUMMARY
ADP6                 | total=   2581 | unique=   2580
antibacterial        | total=  39448 | unique=  39448
DBAASP               | total=   1976 | unique=   1762
dbAMP3               | total=  35599 | unique=  35471


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>